In [1]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix

In [3]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix
from scipy.stats import mode

# load merged dataset
df = pd.read_csv('merged_matrix.csv')

# keep only Room 203 and 204 samples
df_filtered = df[df['roomNo'].isin(['203', '204'])].copy()

# separate features and labels
X = df_filtered.drop(columns=['entryId', 'roomNo'])
y_true = df_filtered['roomNo'].values

# scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# encode labels numerically
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y_true)

# apply KMeans clustering
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
y_pred = kmeans.fit_predict(X_scaled)

# align cluster labels to true labels
mode_label = mode(y_encoded[y_pred == 0], keepdims=True).mode[0]
if mode_label != 0:
    y_pred = 1 - y_pred  # flip labels if reversed

# evaluate accuracy
accuracy = accuracy_score(y_encoded, y_pred)
cm = confusion_matrix(y_encoded, y_pred)

print("✅ K-Means Clustering Results")
print(f"Accuracy: {accuracy * 100:.2f}%")
print("Confusion Matrix:")
print(cm)

print("\nCluster sizes:")
print(pd.Series(y_pred).value_counts())


✅ K-Means Clustering Results
Accuracy: 100.00%
Confusion Matrix:
[[38  0]
 [ 0 34]]

Cluster sizes:
0    38
1    34
Name: count, dtype: int64


In [5]:
import numpy as np

# load merged dataset
df = pd.read_csv('merged_matrix.csv')

# keep only Outside, 203, 204
df_filtered = df[df['roomNo'].isin(['Outside', '203', '204'])].copy()

# separate features and labels
X = df_filtered.drop(columns=['entryId', 'roomNo'])
y_true = df_filtered['roomNo'].values

# scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# encode labels numerically (e.g., Outside=0, 203=1, 204=2)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y_true)

# apply KMeans clustering with 3 clusters
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
y_pred = kmeans.fit_predict(X_scaled)

# align cluster labels to true labels (using majority vote mapping)
label_mapping = {}
for cluster in range(3):
    cluster_labels = y_encoded[y_pred == cluster]
    if len(cluster_labels) == 0:
        label_mapping[cluster] = -1  # empty cluster
    else:
        label_mapping[cluster] = mode(cluster_labels, keepdims=True).mode[0]

# remap predicted clusters to best matching true labels
y_pred_mapped = np.array([label_mapping[c] for c in y_pred])

# calculate accuracy
accuracy = accuracy_score(y_encoded, y_pred_mapped)
cm = confusion_matrix(y_encoded, y_pred_mapped)

print("✅ K-Means (3 Clusters) Results")
print(f"Accuracy: {accuracy * 100:.2f}%")
print("Confusion Matrix:")
print(cm)

print("\nCluster → Label mapping:")
for c, label in label_mapping.items():
    if label != -1:
        print(f"Cluster {c} → {label_encoder.inverse_transform([label])[0]}")
    else:
        print(f"Cluster {c} → (empty)")

✅ K-Means (3 Clusters) Results
Accuracy: 75.59%
Confusion Matrix:
[[37  1  0]
 [ 0 29  5]
 [ 7 18 30]]

Cluster → Label mapping:
Cluster 0 → 204
Cluster 1 → Outside
Cluster 2 → 203
